# Benchmark LLM judge CSVs against human mean scores

This notebook benchmarks every `sophie_1.0_3E_*.csv` file in `data/` against the human reviewer mean for:

- EMPOWER
- EXPLICIT
- EMPATHY
- ALL
- MEAN3E (average of the first three)

It follows the same normalization logic used in your current analysis notebook.


In [1]:
import pandas as pd
import numpy as np
from pathlib import Path

## Paths

This assumes your notebook is in the project root and the data files are inside `data/`.


In [2]:
DATA_DIR = Path("data")
HUMAN_FILE = DATA_DIR / "merged_by_ty.csv"

sorted([p.name for p in DATA_DIR.glob("sophie_1.0_3E_*.csv")])

['sophie_1.0_3E_gpt-4.1-mini_None.csv',
 'sophie_1.0_3E_gpt-4.1_None.csv',
 'sophie_1.0_3E_gpt-5-mini_high.csv',
 'sophie_1.0_3E_gpt-5-mini_low.csv',
 'sophie_1.0_3E_gpt-5.4-mini_high.csv',
 'sophie_1.0_3E_gpt-5.4-mini_low.csv',
 'sophie_1.0_3E_gpt-5.4_high.csv',
 'sophie_1.0_3E_gpt-5.4_low.csv',
 'sophie_1.0_3E_gpt-5_high.csv',
 'sophie_1.0_3E_gpt-5_low.csv']

## Load human ratings

In [3]:
merged_by_ty = pd.read_csv(HUMAN_FILE)
merged_by_ty.head()

,VIDEO_NUM,P_ID,ARM,FIRST_SECOND,CASE_TITLE,P_BACKGROUND,SP_NAME,REVIEWER_NAME,REVIEWER_TYPE,Q1,...,Q14,Q15,Q16,Q17,Q18,Q19,QSUM_EMPOWER,QSUM_EXPLICIT,QSUM_EMPATHY,QSUM_ALL
0,video1829885929,1,S,First,Pat Smith,Student,SP11,SP11,SP,1,...,4,4,1,5,8,4,0.400,0.885714,0.733333,0.634783
1,video8829885929,1,S,Second,Lois Bell,Student,SP11,SP11,SP,2,...,3,4,4,5,7,5,0.675,1.000000,0.766667,0.782609
2,video1757986894,0,C,First,Lois Bell,PA,SP0,SP0,SP,5,...,5,5,4,5,10,10,0.675,0.857143,0.966667,0.834783
3,video2757986894,0,C,Second,Pat Smith,PA,SP0,SP0,SP,5,...,5,5,4,5,10,10,0.875,0.885714,0.966667,0.913043
4,video1631554860,2,S,First,Jill Cooper,Student,SP4,SP4,SP,5,...,2,3,5,2,2,2,0.550,0.828571,0.466667,0.582609


## Collapse SP reviewers and keep required columns

In [4]:
sp_tp_data = merged_by_ty[
    ["P_ID", "ARM", "FIRST_SECOND", "REVIEWER_NAME",
     "QSUM_EMPOWER", "QSUM_EXPLICIT", "QSUM_EMPATHY", "QSUM_ALL"]
].copy()

sp_tp_data["UNIQUE_REVIEWER"] = sp_tp_data["REVIEWER_NAME"].str.replace(
    r"SP\d+", "SP", regex=True
)

sp_tp_data = sp_tp_data.drop(columns=["REVIEWER_NAME"])
sp_tp_data.head()

,P_ID,ARM,FIRST_SECOND,QSUM_EMPOWER,QSUM_EXPLICIT,QSUM_EMPATHY,QSUM_ALL,UNIQUE_REVIEWER
0,1,S,First,0.400,0.885714,0.733333,0.634783,SP
1,1,S,Second,0.675,1.000000,0.766667,0.782609,SP
2,0,C,First,0.675,0.857143,0.966667,0.834783,SP
3,0,C,Second,0.875,0.885714,0.966667,0.913043,SP
4,2,S,First,0.550,0.828571,0.466667,0.582609,SP


## Average repeated ratings within reviewer type

In [5]:
grouped_mean = (
    sp_tp_data
    .groupby(["P_ID", "ARM", "FIRST_SECOND", "UNIQUE_REVIEWER"], as_index=False)
    .mean()
)

grouped_mean.head()

,P_ID,ARM,FIRST_SECOND,UNIQUE_REVIEWER,QSUM_EMPOWER,QSUM_EXPLICIT,QSUM_EMPATHY,QSUM_ALL
0,0,C,First,SP,0.675,0.857143,0.966667,0.834783
1,0,C,First,TP0,0.800,0.885714,0.900000,0.852174
2,0,C,First,TP1,0.800,0.914286,0.933333,0.886957
3,0,C,First,TP2,0.850,0.942857,1.000000,0.930435
4,0,C,First,TP3,0.475,0.828571,0.733333,0.678261


## Pivot to wide format and normalize human reviewer columns

In [6]:
value_cols = ["QSUM_EMPOWER", "QSUM_EXPLICIT", "QSUM_EMPATHY", "QSUM_ALL"]
reviewers = ["SP", "TP0", "TP1", "TP2", "TP3"]

grouped_mean_wide = grouped_mean.pivot_table(
    index=["P_ID", "ARM", "FIRST_SECOND"],
    columns="UNIQUE_REVIEWER",
    values=value_cols,
    aggfunc="first"
)

grouped_mean_wide.columns = [
    f"{metric}_{reviewer}" for metric, reviewer in grouped_mean_wide.columns
]
grouped_mean_wide = grouped_mean_wide.reset_index()

human_cols_to_normalize = [
    "QSUM_ALL_SP", "QSUM_ALL_TP0", "QSUM_ALL_TP1", "QSUM_ALL_TP2", "QSUM_ALL_TP3",
    "QSUM_EMPATHY_SP", "QSUM_EMPATHY_TP0", "QSUM_EMPATHY_TP1", "QSUM_EMPATHY_TP2", "QSUM_EMPATHY_TP3",
    "QSUM_EMPOWER_SP", "QSUM_EMPOWER_TP0", "QSUM_EMPOWER_TP1", "QSUM_EMPOWER_TP2", "QSUM_EMPOWER_TP3",
    "QSUM_EXPLICIT_SP", "QSUM_EXPLICIT_TP0", "QSUM_EXPLICIT_TP1", "QSUM_EXPLICIT_TP2", "QSUM_EXPLICIT_TP3"
]

for col in human_cols_to_normalize:
    grouped_mean_wide[col] = pd.to_numeric(grouped_mean_wide[col], errors="coerce")
    min_val = grouped_mean_wide[col].min()
    max_val = grouped_mean_wide[col].max()
    den = max_val - min_val

    if pd.notna(den) and den != 0:
        grouped_mean_wide[col] = (grouped_mean_wide[col] - min_val) / den
    else:
        grouped_mean_wide[col] = 0.0

for metric in value_cols:
    reviewer_cols = [f"{metric}_{r}" for r in reviewers]
    grouped_mean_wide[f"{metric}_MEAN"] = grouped_mean_wide[reviewer_cols].mean(axis=1)

grouped_mean_wide.head()

,P_ID,ARM,FIRST_SECOND,QSUM_ALL_SP,QSUM_ALL_TP0,QSUM_ALL_TP1,QSUM_ALL_TP2,QSUM_ALL_TP3,QSUM_EMPATHY_SP,QSUM_EMPATHY_TP0,...,QSUM_EMPOWER_TP3,QSUM_EXPLICIT_SP,QSUM_EXPLICIT_TP0,QSUM_EXPLICIT_TP1,QSUM_EXPLICIT_TP2,QSUM_EXPLICIT_TP3,QSUM_EMPOWER_MEAN,QSUM_EXPLICIT_MEAN,QSUM_EMPATHY_MEAN,QSUM_ALL_MEAN
0,0,C,First,0.776471,0.873563,0.964912,0.987342,0.683333,0.958333,0.88,...,0.48,0.791667,0.846154,0.866667,0.913043,0.750,0.742040,0.833506,0.895987,0.857124
1,0,C,Second,0.882353,0.977011,0.982456,0.924051,0.750000,0.958333,1.00,...,0.68,0.833333,0.961538,1.000000,0.782609,0.625,0.800525,0.840496,0.949561,0.903174
2,1,S,First,0.505882,0.448276,0.543860,0.240506,0.383333,0.666667,0.44,...,0.24,0.833333,0.653846,0.400000,0.521739,0.500,0.264614,0.581784,0.471880,0.424372
3,1,S,Second,0.705882,0.356322,0.438596,0.316456,0.450000,0.708333,0.40,...,0.04,1.000000,0.461538,0.533333,0.652174,0.750,0.298867,0.679409,0.455170,0.453451
4,2,S,First,0.435294,0.517241,0.789474,0.848101,0.650000,0.333333,0.20,...,0.48,0.750000,0.615385,0.600000,0.913043,0.625,0.639668,0.700686,0.575419,0.648022


## Helper functions

In [7]:
def minmax_normalize(series):
    series = pd.to_numeric(series, errors="coerce")
    min_val = series.min()
    max_val = series.max()
    den = max_val - min_val

    if pd.notna(den) and den != 0:
        return (series - min_val) / den
    return pd.Series(0.0, index=series.index)

def prepare_llm_file(csv_path):
    llm_3E = pd.read_csv(csv_path)

    llm_3E = llm_3E[
        ["participant", "control/sophie", "pre/post",
         "LLM_SUM_EMPOWER", "LLM_SUM_EXPLICIT", "LLM_SUM_EMPATHY", "LLM_SUM_ALL"]
    ].copy()

    llm_3E = llm_3E.rename(columns={
        "participant": "P_ID",
        "control/sophie": "ARM",
        "pre/post": "FIRST_SECOND"
    })

    llm_3E["P_ID"] = llm_3E["P_ID"].astype(str).str.replace("p", "", regex=False).astype(int)
    llm_3E["ARM"] = llm_3E["ARM"].replace({"control": "C", "sophie": "S"})
    llm_3E["FIRST_SECOND"] = llm_3E["FIRST_SECOND"].replace({"pre": "First", "post": "Second"})

    for col in ["LLM_SUM_EMPOWER", "LLM_SUM_EXPLICIT", "LLM_SUM_EMPATHY", "LLM_SUM_ALL"]:
        llm_3E[col] = minmax_normalize(llm_3E[col])

    return llm_3E

def benchmark_one_file(csv_path, human_df):
    metrics = ["EMPOWER", "EXPLICIT", "EMPATHY", "ALL"]

    llm_3E = prepare_llm_file(csv_path)

    merged_inner = pd.merge(
        llm_3E,
        human_df,
        on=["P_ID", "ARM", "FIRST_SECOND"],
        how="inner"
    ).dropna()

    row = {
        "file": csv_path.name,
        "model": csv_path.stem.replace("sophie_1.0_3E_", ""),
        "n": len(merged_inner),
    }

    for metric in metrics:
        human_col = f"QSUM_{metric}_MEAN"
        llm_col = f"LLM_SUM_{metric}"

        tmp = merged_inner[[human_col, llm_col]].dropna()

        if len(tmp) > 1:
            row[f"{metric}_pearson"] = tmp[human_col].corr(tmp[llm_col], method="pearson")
            # row[f"{metric}_spearman"] = tmp[human_col].corr(tmp[llm_col], method="spearman")
        else:
            row[f"{metric}_pearson"] = np.nan
            # row[f"{metric}_spearman"] = np.nan

        if len(tmp) > 0:
            row[f"{metric}_mae"] = np.mean(np.abs(tmp[human_col] - tmp[llm_col]))
        else:
            row[f"{metric}_mae"] = np.nan

    # row["MEAN3E_pearson"] = np.nanmean([
    #     row["EMPOWER_pearson"], row["EXPLICIT_pearson"], row["EMPATHY_pearson"]
    # ])
    # row["MEAN3E_spearman"] = np.nanmean([
    #     row["EMPOWER_spearman"], row["EXPLICIT_spearman"], row["EMPATHY_spearman"]
    # ])
    # row["MEAN3E_mae"] = np.nanmean([
    #     row["EMPOWER_mae"], row["EXPLICIT_mae"], row["EMPATHY_mae"]
    # ])

    return row

## Benchmark all model CSVs

In [8]:
csv_files = sorted(DATA_DIR.glob("sophie_1.0_3E_*.csv"))
results = pd.DataFrame([benchmark_one_file(csv_path, grouped_mean_wide) for csv_path in csv_files])

results["rank_score"] = (
    results["ALL_pearson"].rank(ascending=False, method="min")
    + results["ALL_mae"].rank(ascending=True, method="min")
)

# Split model and reasoning_effort
results[["model", "reasoning_effort"]] = results["model"].str.rsplit("_", n=1, expand=True)
results["reasoning_effort"] = results["reasoning_effort"].replace("None", "-")

# Optional: reorder columns (put reasoning_effort after model)
cols = results.columns.tolist()
model_idx = cols.index("model")
cols.insert(model_idx + 1, cols.pop(cols.index("reasoning_effort")))
results = results[cols]

results = results.sort_values(
    ["model", "reasoning_effort"], # , "MEAN3E_pearson"
    ascending=[True, False]
).reset_index(drop=True)

results.round(4)

,file,model,reasoning_effort,n,EMPOWER_pearson,EMPOWER_mae,EXPLICIT_pearson,EXPLICIT_mae,EMPATHY_pearson,EMPATHY_mae,ALL_pearson,ALL_mae,rank_score
0,sophie_1.0_3E_gpt-4.1_None.csv,gpt-4.1,-,94,0.7774,0.1190,0.5984,0.1380,0.6970,0.1745,0.7739,0.1077,3.0
1,sophie_1.0_3E_gpt-4.1-mini_None.csv,gpt-4.1-mini,-,94,0.6664,0.1310,0.4949,0.1463,0.6262,0.1676,0.7100,0.1403,18.0
2,sophie_1.0_3E_gpt-5_low.csv,gpt-5,low,94,0.7461,0.1107,0.4843,0.1734,0.6982,0.1380,0.7732,0.1289,8.0
3,sophie_1.0_3E_gpt-5_high.csv,gpt-5,high,94,0.7324,0.1216,0.4663,0.1777,0.6703,0.1418,0.7523,0.1326,12.0
4,sophie_1.0_3E_gpt-5-mini_low.csv,gpt-5-mini,low,94,0.7391,0.1158,0.4840,0.1493,0.5783,0.1866,0.7143,0.1154,11.0
5,sophie_1.0_3E_gpt-5-mini_high.csv,gpt-5-mini,high,94,0.7615,0.1313,0.4768,0.2158,0.6533,0.1558,0.7605,0.1537,13.0
6,sophie_1.0_3E_gpt-5.4_low.csv,gpt-5.4,low,94,0.7993,0.0981,0.4255,0.1695,0.6703,0.1358,0.7586,0.1064,5.0
7,sophie_1.0_3E_gpt-5.4_high.csv,gpt-5.4,high,94,0.7829,0.1279,0.4145,0.1605,0.6615,0.1373,0.7461,0.1259,11.0
8,sophie_1.0_3E_gpt-5.4-mini_low.csv,gpt-5.4-mini,low,94,0.6654,0.1301,0.4709,0.1524,0.5090,0.2304,0.6413,0.1397,18.0
9,sophie_1.0_3E_gpt-5.4-mini_high.csv,gpt-5.4-mini,high,94,0.7594,0.1273,0.4521,0.1490,0.6355,0.1510,0.7320,0.1161,11.0


In [9]:
results_table = results.copy()
results_table = results_table.drop(columns=['file', 'n', 'rank_score'])
results_table = results_table.to_latex(
    index=False,
    escape=True,
    float_format="%.3f"
)
print(results_table)

\begin{tabular}{llrrrrrrrr}
\toprule
model & reasoning\_effort & EMPOWER\_pearson & EMPOWER\_mae & EXPLICIT\_pearson & EXPLICIT\_mae & EMPATHY\_pearson & EMPATHY\_mae & ALL\_pearson & ALL\_mae \\
\midrule
gpt-4.1 & - & 0.777 & 0.119 & 0.598 & 0.138 & 0.697 & 0.174 & 0.774 & 0.108 \\
gpt-4.1-mini & - & 0.666 & 0.131 & 0.495 & 0.146 & 0.626 & 0.168 & 0.710 & 0.140 \\
gpt-5 & low & 0.746 & 0.111 & 0.484 & 0.173 & 0.698 & 0.138 & 0.773 & 0.129 \\
gpt-5 & high & 0.732 & 0.122 & 0.466 & 0.178 & 0.670 & 0.142 & 0.752 & 0.133 \\
gpt-5-mini & low & 0.739 & 0.116 & 0.484 & 0.149 & 0.578 & 0.187 & 0.714 & 0.115 \\
gpt-5-mini & high & 0.761 & 0.131 & 0.477 & 0.216 & 0.653 & 0.156 & 0.761 & 0.154 \\
gpt-5.4 & low & 0.799 & 0.098 & 0.426 & 0.170 & 0.670 & 0.136 & 0.759 & 0.106 \\
gpt-5.4 & high & 0.783 & 0.128 & 0.414 & 0.160 & 0.661 & 0.137 & 0.746 & 0.126 \\
gpt-5.4-mini & low & 0.665 & 0.130 & 0.471 & 0.152 & 0.509 & 0.230 & 0.641 & 0.140 \\
gpt-5.4-mini & high & 0.759 & 0.127 & 0.452 & 0.149 & 0